### **Imports:**
Hieronder zijn alle de packages die ik nodig heb om mijn recomonder model te bouwen.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
import pandas as pd
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\irake\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### **NLP:**
Hieronder pas ik NLP toe op mijn dataset om het goed en lees baar te maken voor de computer (models).

In [ ]:
dataset = pd.read_csv('VKM_dataset_cleaned.csv')

#de kolom combined_text maken door alle tekstkolommen samen te voegen die nodig zijn. 
dataset['combined_text'] = (
    dataset['name'].fillna('') + ' ' +
    dataset['shortdescription'].fillna('') + ' ' +
    dataset['description'].fillna('') + ' ' +
    dataset['content'].fillna('') + ' ' +
    dataset['learningoutcomes'].fillna('') + ' ' +
    dataset['module_tags'].fillna('')
)

# stopword functie aanroepen
stop_words = set(stopwords.words('dutch'))
preprocessed_texts = []

#Voor elke rij in combined_text kolom NLP uitvoeren.
for text in dataset['combined_text']:
    #Tekst lowercase maken
    text_lower = text.lower()
    #alle niet relevante nummers uit de tekst halen.
    text_no_numbers = re.sub(r'\d+', '', text_lower)
    #alle leestekens uit de tekst halen.
    text_no_punct = re.sub(r'[^\w\s]', '', text_no_numbers)
    
    #alle woorden appart maken om de stop woorden eruit te halen.
    tokens = []
    for sentence in sent_tokenize(text_no_punct):
        words = word_tokenize(sentence)
        filtered = [w for w in words if w not in stop_words]
        tokens.extend(filtered)
    
    #Tokanization weer terugdraaien, dus de ipv apparte woorden weer terug naar een gehele zin. 
    preprocessed_texts.append(' '.join(tokens))

### **Content-based Recommender model:**
Nadat ik NLP heb uitgevoerd heb ik hieronder de contend-based recommender gebouwed met de model if-idf. 

In [ ]:
vectorizer = TfidfVectorizer()
#bekijkt elke document(rij) en geeft van elke rij de unieke waarden een score.
tfidf_matrix = vectorizer.fit_transform(preprocessed_texts)
#Hier maak ik een nieuwe kolom om alle if-idf scores van 1 rij op te sommen en in de nieuwe kolom te stoppen. 
dataset['tfidf_score'] = tfidf_matrix.sum(axis=1).A1 

### **Hybride Model:**
Hieronder pas ik de hybride model toe op mijn dataset die een top 5 keuzemodules geeft op basis van de tf-idf score, interest en popularity score. 

In [ ]:
scaler = MinMaxScaler()

#Hier maak ik 3 nieuwe kolommen waarvan de interest, popularity en tf-idf score worden gescaled tussen -1 en 1 voor elke rij. 
dataset['interest_match_norm'] = scaler.fit_transform(dataset['interests_match_score'].values.reshape(-1,1))
dataset['popularity_score_norm'] = scaler.fit_transform(dataset['popularity_score'].values.reshape(-1,1))
dataset['tfidf_score_norm'] = scaler.fit_transform(dataset['tfidf_score'].values.reshape(-1,1))
#Hier ga ik een nieuwe kolom maken waarvan ik bepaal hoe zwaar elke onderdeel voor elke rij meeteld en dan tel ik het bij elkaar op.
dataset['hybrid_score'] = 0.5 * dataset['interest_match_norm'] + 0.1 * dataset['popularity_score_norm'] + 0.4 * dataset['tfidf_score_norm']

#Top 5 uitprinten, dus van de hybride_score kolom de 5 keuzemodules waarvan de score het hoogste is laat ik hierzo zien.
top5_hybrid = dataset.sort_values(by='hybrid_score', ascending=False).head(5)

print('top 5 keuzemodules:\n', top5_hybrid[['name']])

top 5 keuzemodules:
                                                   name
198                                         stopmotion
182  multdisciplinair samenwerken in een beroepscon...
118                   robotic ai interfaces - optie 2*
51          act for change together nlqf6 30 + 15 ects
112                                    robot challenge


### **student Input:**
Hieronder voer ik een student input toe waarna ik de tekst van de student input process om te gebruiken voor het model.

In [ ]:
#Input van een student.
student_input = "docent, gamen, fitness, koken"

#NLP uitvoeren voor de input van de student.
student_profile_lower_text = student_input.lower()
student_profile_no_numbers = re.sub(r'\d+', '', student_profile_lower_text)
student_profile_no_punct = re.sub(r'[^\w\s]', '', student_profile_no_numbers)
tokens = []
for sentence in sent_tokenize(student_profile_no_punct):
    words = word_tokenize(sentence)
    filtered = [w for w in words if w not in stop_words]
    tokens.extend(filtered)

#De woorden weer tot een zin maken.
student_text = ' '.join(tokens)

#De input krijgt tf-idf score op basis van de eerder getrainde dataset en dat is dus wat er in de tfidfmatrix zit. dus nieuwe woorden krijg score van 0 en woorden die in de dataset komen met deze scoren krijgen een hoger score. 
student_vector = vectorizer.transform([student_text])

### **cosine** 
gebruik maken van cosin om de tfdf dus hoevaak iets voorkomt in de dataset te vergelijken met de student vector om dan een top 5 te genereren op de basis van de cosin similarity.

In [ ]:
#hier wordt gekeken welke woorden in student_vector en tf-idf_matrix heel vergelijkbaar met elkaar zijn en die krijgt een score tussen -1 en 1. 
similarities = cosine_similarity(student_vector, tfidf_matrix)

#Je gaat hier basically van arrays in één array naar 1 array toe. [[1], [2]] -> [1, 2]
similarities = similarities.flatten()
#Pakt de top 5 modules
top_indices = similarities.argsort()[::-1][:5]  

#laat de top rijen zien.
recommended_modules = dataset.iloc[top_indices]
recommended_modules[['name', 'shortdescription', 'module_tags']]

,name,shortdescription,module_tags
179,didactiek voor n&g en n&t,"didactiek, docentschap, natuur, techniek, gezo...","didactiek, docentschap, natuur, techniek, gezo..."
167,forensische chemie,"chemische analyse, data analyse, forensic scie...","chemische, analyse, data, analyse, forensic, s..."
210,de stem van je geweten. ga opzoek naar jouw mo...,"moreel kompas, ethische dillema's, normen & wa...","moreel, kompas, ethische, dillemas, normen, wa..."
207,avans innovative studio junior,"persoonlijke ontwikkeling, interdisciplinair, ...","persoonlijke, ontwikkeling, interdisciplinair,..."
206,ethiek & kritisch denken,"ethiek, filosofie, kritisch denken, ethisch ha...","ethiek, filosofie, kritisch, denken, ethisch, ..."


In [ ]:
feature_names = vectorizer.get_feature_names_out()

for i in top_indices:
    doc_vector = tfidf_matrix[i].toarray()[0]
    word_scores = dict(zip(feature_names, doc_vector))
    top_words = sorted(word_scores.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"Module: {dataset.iloc[i]['name']}")
    print("Waarom passend? Belangrijkste woorden:", [w[0] for w in top_words])
    print()

Module: didactiek voor n&g en n&t
Waarom passend? Belangrijkste woorden: ['docent', 'natuur', 'techniek', 'didactiek', 'gezondheid']

Module: forensische chemie
Waarom passend? Belangrijkste woorden: ['analyse', 'science', 'chemische', 'forensische', 'data']

Module: de stem van je geweten. ga opzoek naar jouw moreel kompas.
Waarom passend? Belangrijkste woorden: ['ethisch', 'dilemmas', 'kompas', 'moreel', 'ethische']

Module: avans innovative studio junior
Waarom passend? Belangrijkste woorden: ['kijker', 'valorisatie', 'persoonlijke', 'interdisciplinair', 'prototyping']

Module: ethiek & kritisch denken
Waarom passend? Belangrijkste woorden: ['handelen', 'ethisch', 'ethiek', 'beeld', 'vakgebied']

